Benchmark
============

Adding all planners
===========


In [ ]:
import matplotlib.pylab as plt
%matplotlib inline
from IPPerfMonitor import IPPerfMonitor
#import IPBasicPRM
#import IPVISBasicPRM

#import IPVisibilityPRM
#import IPVISVisibilityPRM

#import IPLazyPRM
#import IPVISLazyPRM

#import IPRRT
#import IPVISRRT

import IPAStar
import IPVISAStar
import AStarReopeningForShapeRobot

Set-up of the test scenario and the configuration for all planner
===================================

Following a procedure to compare all discussed planners are shown:

1. Configuration for every planner is defined
2. The configuration and the planner is stored in the variable setup, a Python dict()
3. the variable setup is then used to uniformly call the planning calls


In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML, display
import copy
import numpy as np
import matplotlib.pyplot as plt

# Hilfsfunktion (unverändert, aber wir behandeln den Rückgabewert jetzt besser)
def _interpolate_line(startPos, endPos, step_l):
    steps = []
    line = np.array(endPos) - np.array(startPos)
    line_l = np.linalg.norm(line)
    if line_l == 0:
        return [np.array(startPos)]
    step = line / line_l * step_l
    n_steps = int(np.floor(line_l / step_l))
    c_step = np.array(startPos, dtype=float)
    for i in range(n_steps):
        steps.append(copy.deepcopy(c_step))
        c_step = c_step + step
    if not np.allclose(c_step, np.array(endPos)):
        steps.append(np.array(endPos, dtype=float))
    return steps

def animate_solution_multi_downsampled(prm, environment, solution, visualizer=None, 
                                     workSpaceLimits=None, step_l=0.1, interval=50, 
                                     max_frames=200, figsize=(6,6), dpi=40, 
                                     pause_frames=40):
    
    # 1. Basic Checks
    if not solution or len(solution) < 2:
        raise ValueError('solution must contain at least two nodes')

    # 1. Récupération des positions pour l'interpolation du robot
    path_pos = [prm.graph.nodes[n]['pos'] for n in solution]
    
    # Interpolation (Mouvement fluide du point rouge)
    full_path = [path_pos[0]]
    for i in range(1, len(path_pos)):
        s, e = np.array(path_pos[i-1]), np.array(path_pos[i])
        dist = np.linalg.norm(e - s)
        if dist > 0:
            n_steps = int(np.floor(dist / step_l))
            for j in range(1, n_steps + 1):
                full_path.append(s + (e - s) * (j / (n_steps + 1)))
    full_path.append(path_pos[-1])

    # Downsampling
    if len(full_path) > max_frames:
        indices = np.linspace(0, len(full_path)-1, max_frames).astype(int)
        frames_data = [full_path[i] for i in indices]
    else:
        frames_data = full_path

    # 2. Setup Figure
    fig_local, ax = plt.subplots(figsize=figsize, dpi=dpi)
    limits = environment.getEnvironmentLimits()

    def _animate(current_pos):
        ax.cla() # On efface pour redessiner proprement
        
        # --- VISUALIZER OHNE ROBOTER-SHAPES ---
        # Deaktiviere temporär die drawRobot Methode, damit der Visualizer
        # die Diskretisierung zeichnet, aber nicht die grünen Roboter-Shapes
        original_drawRobot = environment.robot.drawRobot
        environment.robot.drawRobot = lambda ax, **kwargs: None  # Dummy-Funktion
        
        visualizer(prm, solution, ax=ax, nodeSize=100)
        
        # Stelle die ursprüngliche drawRobot Methode wieder her
        environment.robot.drawRobot = original_drawRobot
        
        # On redessine les obstacles par sécurité
        environment.drawObstacles(ax)
        
        # On remet les limites (parce que visualizer peut les changer)
        ax.set_xlim(limits[0])
        ax.set_ylim(limits[1])
        ax.set_title("A* Visualization & Animation")

        # --- MARKIERUNG VON START UND ZIEL ---
        # Startpunkt: Grüner Punkt mit 'S'
        start_pos = path_pos[0]
        ax.plot(start_pos[0], start_pos[1], 'o', color='#00dd00', 
                markersize=15, markeredgecolor='black', zorder=11)
        ax.text(start_pos[0], start_pos[1], 'S', ha='center', va='center',
                fontsize=12, fontweight='bold', color='black', zorder=12)
        
        # Zielpunkt: Roter Punkt mit 'G'
        goal_pos = path_pos[-1]
        ax.plot(goal_pos[0], goal_pos[1], 'o', color='#DD0000', 
                markersize=15, markeredgecolor='black', zorder=11)
        ax.text(goal_pos[0], goal_pos[1], 'G', ha='center', va='center',
                fontsize=12, fontweight='bold', color='white', zorder=12)

        # --- NUR DER ANIMIERTE ROTE ROBOTER ---
        environment.robot.setTo(current_pos)
        environment.robot.drawRobot(ax, color='red', alpha=0.9)

    print(f"Rendu de l'animation (Diskretisierung + nur roter Roboter)...")
    ani = animation.FuncAnimation(fig_local, _animate, frames=frames_data, interval=interval)
    
    html = HTML(ani.to_jshtml())
    display(html)
    plt.close()
    return ani

In [ ]:
plannerFactory = dict()

#basicConfig = dict()
#basicConfig["radius"] = 3
#basicConfig["numNodes"] = 200
#plannerFactory["basePRM"] = [IPBasicPRM.BasicPRM, basicConfig, IPVISBasicPRM.basicPRMVisualize]

#visbilityConfig = dict()
#visbilityConfig["ntry"] = 300
#plannerFactory["visibilityPRM"] = [IPVisibilityPRM.VisPRM, visbilityConfig, IPVISVisibilityPRM.visibilityPRMVisualize ]

# LazyPRM funktioniert nicht mit 3D-Konfigurationsraum (x, y, theta)
# Nur für 2D-Roboter ohne Orientierung geeignet

# ========================================
# A* Variante 1: w=0.5, grobe Diskretisierung
# ========================================
astarConfig1 = dict()
astarConfig1["heuristic"] = 'euclidean' 
astarConfig1["w"] = 0.5
astarConfig1["num_steps"] = [20, 20, 36]  # grob
plannerFactory["astar_w05_grobe_diskretisierung"] = [IPAStar.AStar, astarConfig1, IPVISAStar.aStarVisualizeWspace]

# ========================================
# A* Variante 2: w=0.7, grobe Diskretisierung
# ========================================
astarConfig2 = dict()
astarConfig2["heuristic"] = 'euclidean' 
astarConfig2["w"] = 0.7
astarConfig2["num_steps"] = [20, 20, 36]  # grob
plannerFactory["astar_w07_grobe_diskretisierung"] = [IPAStar.AStar, astarConfig2, IPVISAStar.aStarVisualizeWspace]

# # ========================================
# # A* Variante 3: w=0.5, feine Diskretisierung
# # ========================================
astarConfig3 = dict()
astarConfig3["heuristic"] = 'euclidean' 
astarConfig3["w"] = 0.5
astarConfig3["num_steps"] = [44, 44, 36]  # feiner
plannerFactory["astar_w05_feine_diskretisierung"] = [IPAStar.AStar, astarConfig3, IPVISAStar.aStarVisualizeWspace]

# # ========================================
# # A* Variante 4: w=0.7, feine Diskretisierung
# # ========================================
# astarConfig4 = dict()
# astarConfig4["heuristic"] = 'euclidean' 
# astarConfig4["w"] = 0.7
# astarConfig4["num_steps"] = [44, 44, 36]  # feiner
# plannerFactory["astar_w07_fine"] = [IPAStar.AStar, astarConfig4, IPVISAStar.aStarVisualizeWspace]

# ========================================
# A* Variante 3: Mit Edge Collision Check
# ========================================
astarConfig3 = dict()
astarConfig3["heuristic"] = 'euclidean' 
astarConfig3["w"] = 0.5
astarConfig3["num_steps"] = [20, 20, 36]
astarConfig3["checkEdgeCollision"] = True  # MIT Kantenkollisionsprüfung
plannerFactory["astar_w05_with_edgeCheck"] = [IPAStar.AStar, astarConfig3, IPVISAStar.aStarVisualizeWspace]

# ========================================
# A* Variante 4: Ohne Edge Collision Check
# ========================================
astarConfig4 = dict()
astarConfig4["heuristic"] = 'euclidean' 
astarConfig4["w"] = 0.5
astarConfig4["num_steps"] = [20, 20, 36]
astarConfig4["checkEdgeCollision"] = False  # OHNE Kantenkollisionsprüfung
plannerFactory["astar_w05_no_edgeCheck"] = [IPAStar.AStar, astarConfig4, IPVISAStar.aStarVisualizeWspace]

# ========================================
# A* MIT Reopening
# ========================================
astarReopen_on = dict()
astarReopen_on["heuristic"] = 'euclidean' 
astarReopen_on["w"] = 0.5
astarReopen_on["num_steps"] = [20, 20, 36]
astarReopen_on["checkEdgeCollision"] = False
plannerFactory["astar_w05_reopening_ON"] = [AStarReopeningForShapeRobot.ReopenAStar, astarReopen_on, IPVISAStar.aStarVisualizeWspace]

# ========================================
# A* OHNE Reopening (zum Vergleich)
# ========================================
astarReopen_off = dict()
astarReopen_off["heuristic"] = 'euclidean' 
astarReopen_off["w"] = 0.5
astarReopen_off["num_steps"] = [20, 20, 36]
astarReopen_off["checkEdgeCollision"] = False
# ReopenAStar mit reopen=False initialisieren
# Anmerkung: Der Constructor nimmt reopen als Parameter, nicht aus config
# Wir verwenden hier die normale IPAStar.AStar für "reopening OFF"
plannerFactory["astar_w05_reopening_OFF"] = [IPAStar.AStar, astarReopen_off, IPVISAStar.aStarVisualizeWspace]


In [ ]:
class ResultCollection (object):
    
    def __init__(self, plannerFactoryName, planner, benchmark, solution, perfDataFrame):
        self.plannerFactoryName = plannerFactoryName
        self.planner = planner
        self.benchmark = benchmark
        self.solution = solution
        self.perfDataFrame = perfDataFrame

In [ ]:
import IPTestSuite2 as ts
from shapely.geometry import Point, Polygon, LineString
from shapely import plotting


In [ ]:
#import importlib
#importlib.reload(IPTestSuiteSS2024)

In [ ]:
fullBenchList = ts.benchList

for benchmark in fullBenchList:
    print(benchmark.name)

In [ ]:
for benchmark in fullBenchList:
    fig_local = plt.figure(figsize=(7,7))
    ax = fig_local.add_subplot(1,1,1)
    title = benchmark.name
    ax.set_title(title)
    ax.set_xlim(benchmark.collisionChecker.getEnvironmentLimits()[0])
    ax.set_ylim(benchmark.collisionChecker.getEnvironmentLimits()[1])
    plotting.plot_points(Point(benchmark.startList[0]).buffer(.3), color="g", ax=ax)
    plotting.plot_points(Point(benchmark.goalList[0]).buffer(.3), color="b", ax=ax)
    #try:
    benchmark.collisionChecker.drawObstacles(ax)
  
        
    #except Exception as e:
    #    print ("Error", e)
    #    pass
    plt.show()

In [ ]:
resultList = list()
testList = fullBenchList

for key,producer in list(plannerFactory.items()):
    print(key, producer)
    for benchmark in testList:
        print ("Planning: " + key + " - " + benchmark.name)
        #planner = IPBasicPRM.BasicPRM(benchmark.collisionChecker)
        planner = producer[0](benchmark.collisionChecker)
        IPPerfMonitor.clearData()
        try:
            
            resultList.append(ResultCollection(key,
                                          planner, 
                                           benchmark, 
                                           planner.planPath(benchmark.startList,benchmark.goalList,producer[1]),
                                           IPPerfMonitor.dataFrame()
                                          ),
                        )
        except Exception as e:
        #    throw e
            print ("PLANNING ERROR ! PLANNING ERROR ! PLANNING ERROR ", e)
            pass



In [ ]:
import matplotlib.pyplot as plt

for result in resultList:
    
    fig_local = plt.figure(figsize=(20,20))
    ax = fig_local.add_subplot(1,1,1)
    title = result.plannerFactoryName + " - " + result.benchmark.name
    if result.solution == []:
        title += " (No path found!)"
    title += "\n Assumed complexity level " + str(result.benchmark.level)
    ax.set_title(title)
    try:
        #IPVISBasicsPRM.basicPRMVisualize(result.planner, result.solution, ax=ax, nodeSize=100))
        plannerFactory[result.plannerFactoryName][2](result.planner, result.solution, ax=ax, nodeSize=100)
    except Exception as e:
        print ("Error", e)
        pass
    
    plt.show()

In [ ]:
for bench in testList:
    for result in resultList:
        if result.benchmark.name == bench.name:
            # Prüfe ob Lösung existiert UND mindestens 2 Knoten hat
            if result.solution is not None and len(result.solution) >= 2:
                print(f"Animation für: {result.plannerFactoryName} - {bench.name}")
                print(f"  Anzahl Knoten im Pfad: {len(result.solution)}")
                try:
                    animate_solution_multi_downsampled(
                        result.planner, bench.collisionChecker, result.solution,
                        visualizer=plannerFactory[result.plannerFactoryName][2],
                        step_l=0.05, interval=60, max_frames=200, figsize=(20,20), dpi=40)
                except Exception as e:
                    print(f"  FEHLER bei Animation: {e}")
            else:
                if result.solution is None:
                    print(f"Keine Animation für {result.plannerFactoryName} - {bench.name}: Keine Lösung gefunden")
                else:
                    print(f"Keine Animation für {result.plannerFactoryName} - {bench.name}: Zu wenige Knoten ({len(result.solution)})")

In [ ]:
import numpy as np
for bench in testList:
    title = bench.name
    pathLength = dict()
    planningTime = dict()
    roadmapSize  = dict()
    
    try:
        for result in resultList:
            if result.benchmark.name == bench.name:
                #print result.benchmark.name  + " - " +  result.plannerFactoryName, len(result.solution)
                pathLength[result.plannerFactoryName] = len(result.solution)
                planningTime[result.plannerFactoryName] = result.perfDataFrame.groupby(["name"]).sum(numeric_only=True)["time"]["planPath"]
                roadmapSize[result.plannerFactoryName] = result.planner.graph.size()


        fig, ax = plt.subplots(figsize=(16, 6))  # Breitere Figur

        width = 0.2

        ax.bar(np.arange(len(pathLength.keys())), pathLength.values(),width, color="blue")
        ax.set_ylabel(title + " Number of nodes in path", color="blue")
        ax.set_xticks(np.arange(len(pathLength.keys())) + width)
        ax.set_xticklabels(pathLength.keys(), rotation=45, ha='right')  # Rotierte Labels

        ax2 = ax.twinx()
        bar = ax2.bar(np.arange(len(pathLength.keys()))+width, planningTime.values(),width, color="red")
        ax2.set_ylabel(title + " Planning time", color="y")

        # Add coloring and patterns on axis two
        hatches = ['x' if length==0 else '' for length in pathLength.values()]
        color   = ['red' if length==0 else 'yellow' for length in pathLength.values()]
        for i,thisbar in enumerate(bar.patches):
            thisbar.set_facecolor(color[i])
            thisbar.set_hatch(hatches[i])

        # Multiple axes 
        ax3 = ax.twinx()
        ax3.bar(np.arange(len(pathLength.keys()))+2*width, roadmapSize.values(),width, color="purple")
        ax3.set_ylabel(title + " Roadmap size",  color="purple")
        ax3.spines['right'].set_position(('axes', 1.15))
        ax3.spines['right'].set_color("purple")
        
        plt.tight_layout()  # Besseres Layout, verhindert abgeschnittene Labels
    except:
        pass

    plt.show()
    